# Лабораторная работа № 2
## Кластеризация учебных траекторий и прогнозирование академического риска

**Дисциплина:** «Методы анализа больших данных» · **Профиль:** «Информатика и английский язык» · **Компетенция:** ПК-1.2 · **Трудоёмкость:** 10 ак. ч · **Максимум:** 20 баллов

**Среда:** Google Colab, Python 3.10+ · **Данные:** UCI Student Performance, `student-mat.csv` (395 учащихся двух португальских школ, предмет — математика)

**Цель.** Пройти полный цикл анализа образовательных данных: подготовить признаки, найти учебные профили методом K-Means, построить модель прогноза итоговой оценки `G3`, честно оценить её качество и превратить результат в индивидуальный образовательный маршрут (ИОМ), который **проверяет и утверждает учитель**.

### Как работать с ноутбуком

Главное в работе — **не код, а понимание и выводы**: что показали кластеры, каким числам модели можно доверять и как использовать прогноз педагогически корректно. Почти весь код готов.

| Что видите в ячейке | Что сделать |
|---|---|
| `V = None` | впишите номер своего варианта (целое число 1–25) |
| `...` (три точки) и справа `# ← впишите: …` | сотрите три точки и впишите код из подсказки (можно скопировать) |
| `Не заполнено (…)` после запуска | в ячейке осталось `...`; в сообщении указано, какая строка |
| **«Ячейка готова — просто запустите»** | ничего не меняйте, нажмите **Shift + Enter** |
| **«Как читать результат»** | прочитайте перед тем, как писать вывод |
| **«Что получено»** | дважды щёлкните по ячейке и допишите предложения своими числами — это основная часть отчёта |

**Порядок.** Выполняйте ячейки сверху вниз. Если появилась ошибка `NameError: name '…' is not defined` — не запущена одна из **предыдущих** ячеек: выберите **Runtime → Run before**. Перед сдачей — **Runtime → Restart and run all**.

| Где дописать код | Что вписать |
|---|---|
| Шаг 0 | ФИО, группа, номер варианта |
| Шаг 3, TODO 1 | масштабирование и кодирование признаков |
| Шаг 4, TODO 2 | выбранное число кластеров `k` |
| Шаг 6, TODO 3–4 | исключение `G3` из признаков и стратифицированное разбиение |

### Критерии оценивания (20 баллов)

| Баллы | Что оценивается | Где в ноутбуке |
|---:|---|---|
| 5 | предобработка, масштабирование и K-Means: выбор `k`, профили кластеров | шаги 2–5 |
| 5 | построение и обучение классификатора своего варианта | шаги 6–7 |
| 4 | матрица ошибок, F1 и интерпретация (сравнение с простым правилом, важность признаков) | шаги 8–9 |
| 4 | педагогический кейс: ИОМ | шаг 10 |
| 2 | защита и контрольные вопросы | на защите |

## Теоретический минимум — коротко и на примерах

### Конвейер работы (CRISP-DM в образовании)

```text
Задача педагога → Данные → Подготовка (масштаб + кодирование) ─┬─ K-Means → профили групп
                                                               └─ Классификатор → прогноз риска
                         → Совместная интерпретация → ИОМ → проверка учителем → мониторинг
```

### K-Means: «раскладываем учеников по похожести»

Алгоритм выбирает `k` центров и относит каждого ученика к ближайшему центру, затем сдвигает центры в середину своих групп и повторяет. Он минимизирует сумму квадратов расстояний до центров:

$$J=\sum_{j=1}^{k}\sum_{x_i\in C_j}\lVert x_i-c_j\rVert^2 .$$

`inertia_` в scikit-learn — это и есть $J$: чем меньше, тем плотнее группы. При росте `k` инерция **всегда** падает, поэтому её одной недостаточно для выбора `k`.

### Зачем масштабировать

K-Means сравнивает расстояния. Пропуски `absences` меняются от 0 до 75, а `studytime` — от 1 до 4. Без масштабирования «победят» пропуски просто потому, что у них больше числа. Стандартизация приводит признаки к одной шкале:

$$z=\frac{x-\mu}{\sigma}.$$

Категориальные признаки (`yes`/`no`, названия школ) переводятся в числа **One-Hot кодированием**: `internet = yes` → два столбца `internet_no = 0`, `internet_yes = 1`.

### Как выбрать k: локоть и силуэт

* **Метод локтя** — ищем место, где кривая инерции перестаёт резко падать.
* **Силуэт** для объекта $i$: $s(i)=\dfrac{b(i)-a(i)}{\max(a(i),b(i))}$, где $a(i)$ — среднее расстояние до своих, $b(i)$ — до ближайшей чужой группы. От −1 до 1: ближе к 1 — группы чёткие; около 0,1–0,2 — группы размытые, это **условное деление**, а не «естественные типы учеников».
* Третий критерий — **интерпретируемость**: сможет ли учитель описать каждую группу словами.

### Цель прогноза

$$risk=\begin{cases}1, & G3<10\\ 0, & G3\ge 10\end{cases}\qquad
grade(G3)=\begin{cases}A, & 16\le G3\le 20\\ B, & 14\le G3\le 15\\ C, & 10\le G3\le 13\\ F, & 0\le G3\le 9\end{cases}$$

Оценки в данных — по 20-балльной португальской шкале; 10 — порог «сдал». Шкала A/B/C/F — учебная, только для этой работы.

### Метрики: какая ошибка дороже

| | прогноз «нет риска» | прогноз «риск» |
|---|---|---|
| **на самом деле нет риска** | TN | FP — лишняя консультация |
| **на самом деле риск** | FN — **ученик остался без помощи** | TP |

$$Precision=\frac{TP}{TP+FP},\quad Recall=\frac{TP}{TP+FN},\quad F_1=\frac{2PR}{P+R},\quad F_1^{macro}=\frac1K\sum_k F_{1,k}.$$

Для системы раннего предупреждения важнее **Recall** класса риска (не пропустить), но при очень низком Precision учитель утонет в ложных сигналах. Accuracy при дисбалансе обманывает: если рискует треть класса, «никто не рискует» уже даёт 67 %.

---
# Шаг 0. Студент и вариант

**Что вписать:** ФИО и группу в кавычках, номер варианта — **целым числом без кавычек** (например, `V = 7`). Всё остальное — признаки, цель, алгоритм и гиперпараметры — подставится из таблицы вариантов автоматически.

**До запуска модели** запишите гипотезу в блок ниже.

### ✍️ Гипотеза (заполните до обучения моделей)

> Какие признаки, по-вашему, сильнее всего связаны с риском неуспеваемости в вашем варианте? Будет ли кластер с низкими `G1`/`G2` совпадать с фактическим риском? Ответьте 1–2 предложениями.
>
> _[ваша гипотеза]_

In [ ]:
# =========== ЗАПОЛНИТЕ ===========
STUDENT_NAME  = "Иванов Иван Иванович"   # TODO: ← впишите свои ФИО (в кавычках)
STUDENT_GROUP = "ГР-00"                  # TODO: ← впишите группу (в кавычках)
V = None                                 # TODO: ← впишите номер варианта, например V = 7
# =================================

VARIANTS = {
    1:  dict(features=["studytime", "failures", "G1", "G2"], target="binary", algo="Decision Tree",
             params=dict(max_depth=3, min_samples_leaf=8), case="Диагностика дефицитов и еженедельный контроль"),
    2:  dict(features=["studytime", "failures", "absences", "G1", "G2"], target="binary", algo="Logistic Regression",
             params=dict(C=0.5, class_weight="balanced"), case="Контроль посещаемости и календарь контрольных точек"),
    3:  dict(features=["freetime", "goout", "Dalc", "Walc", "absences"], target="grade", algo="Random Forest",
             params=dict(n_estimators=200, max_depth=6), case="Саморегуляция учебного времени и мониторинг посещаемости"),
    4:  dict(features=["Medu", "Fedu", "famrel", "famsize", "Pstatus"], target="binary", algo="KNN",
             params=dict(n_neighbors=9, weights="distance"), case="Поддержка без социальных ярлыков, опора на доступные ресурсы"),
    5:  dict(features=["G1", "G2", "failures", "schoolsup", "paid"], target="grade", algo="Decision Tree",
             params=dict(max_depth=5, min_samples_split=12), case="Дифференцированные задания по текущим достижениям"),
    6:  dict(features=["studytime", "higher", "failures", "G1", "G2"], target="binary", algo="Random Forest",
             params=dict(n_estimators=250, max_depth=5, class_weight="balanced"), case="Мотивационный модуль и предметные консультации"),
    7:  dict(features=["absences", "traveltime", "freetime", "goout", "studytime"], target="binary", algo="Logistic Regression",
             params=dict(C=1.0, class_weight="balanced"), case="Планирование времени и контроль посещаемости"),
    8:  dict(features=["schoolsup", "famsup", "paid", "internet", "activities", "G1"], target="grade", algo="Random Forest",
             params=dict(n_estimators=300, max_depth=7), case="Подбор бесплатных цифровых и очных форм поддержки"),
    9:  dict(features=["Medu", "Fedu", "studytime", "failures", "G1"], target="binary", algo="Decision Tree",
             params=dict(max_depth=4, class_weight="balanced"), case="Индивидуальные цели без выводов о способностях по семейным факторам"),
    10: dict(features=["goout", "Walc", "absences", "G1", "G2"], target="grade", algo="KNN",
             params=dict(n_neighbors=11, weights="distance", p=2), case="Анализ режима и регулярности учебной работы"),
    11: dict(features=["studytime", "failures", "absences", "G1"], target="binary", algo="Logistic Regression",
             params=dict(C=0.2, class_weight="balanced"), case="Раннее вмешательство после первого периода", exclude=["G2"]),
    12: dict(features=["school", "reason", "schoolsup", "activities", "internet", "G1"], target="grade", algo="Random Forest",
             params=dict(n_estimators=200, max_depth=8), case="Маршрутизация к школьным ресурсам"),
    13: dict(features=["famrel", "famsup", "Medu", "Fedu", "studytime", "G1"], target="binary", algo="KNN",
             params=dict(n_neighbors=7, weights="uniform", p=1), case="Подбор форм обратной связи и учебных ресурсов"),
    14: dict(features=["freetime", "goout", "Dalc", "Walc", "health", "absences"], target="grade", algo="Decision Tree",
             params=dict(max_depth=4, min_samples_leaf=10), case="Работа с учебным режимом без медицинских интерпретаций"),
    15: dict(features=["studytime", "internet", "higher", "G1", "G2"], target="binary", algo="Random Forest",
             params=dict(n_estimators=250, max_features="sqrt", class_weight="balanced"), case="Офлайн/онлайн ресурсы и задания разного уровня"),
    16: dict(features=["absences", "failures", "schoolsup", "G1", "G2"], target="grade", algo="Logistic Regression",
             params=dict(C=2.0, max_iter=2000), case="Консультации и восстановление пропущенных тем"),
    17: dict(features=["address", "traveltime", "goout", "freetime", "internet"], target="binary", algo="Decision Tree",
             params=dict(max_depth=3, min_samples_leaf=12), case="Реалистичный график и доступность ресурсов"),
    18: dict(features=["G1", "G2", "studytime", "failures", "absences"], target="binary", algo="KNN",
             params=dict(n_neighbors=5, weights="distance", p=2), case="Персональные задания по близким учебным профилям"),
    19: dict(features=["schoolsup", "famsup", "paid", "studytime", "G1", "G2"], target="grade", algo="Random Forest",
             params=dict(n_estimators=400, max_depth=6, min_samples_leaf=2), case="Сопоставление вида поддержки и учебного профиля"),
    20: dict(features=["higher", "studytime", "activities", "goout", "absences", "G1"], target="binary", algo="Logistic Regression",
             params=dict(C=0.8, class_weight="balanced"), case="Учебная цель, план недели и короткая обратная связь"),
    21: dict(features=["studytime", "failures", "famsup", "famrel", "G1", "G2"], target="grade", algo="Decision Tree",
             params=dict(max_depth=6, min_samples_leaf=6), case="Задания разной сложности и формирующее оценивание"),
    22: dict(features=["school", "reason", "traveltime", "studytime", "G1", "G2"], target="binary", algo="Random Forest",
             params=dict(n_estimators=300, max_depth=5, class_weight="balanced_subsample"), case="Адаптация графика и школьная поддержка"),
    23: dict(features=["freetime", "goout", "absences", "failures", "G1", "G2"], target="grade", algo="KNN",
             params=dict(n_neighbors=13, weights="distance", p=2), case="Планирование нагрузки по профилю группы"),
    24: dict(features=["studytime", "failures", "schoolsup", "famsup", "higher", "internet", "G1", "G2"], target="binary",
             algo="Logistic Regression", params=dict(C=1.5, class_weight="balanced", max_iter=2000),
             case="Комбинированная поддержка: контент, консультация, контроль"),
    25: dict(features=["studytime", "failures", "absences", "goout", "Walc", "famrel", "higher", "schoolsup", "famsup",
                       "internet", "G1", "G2"], target="binary", algo="Random Forest",
             params=dict(n_estimators=300, max_depth=6, min_samples_leaf=3, class_weight="balanced"),
             case="Диагностика дефицитов, персональные задания, консультации и двухнедельный мониторинг"),
}

assert V is not None, "Вариант: впишите номер варианта вместо None, например V = 7"
assert isinstance(V, int) and 1 <= V <= 25, "Вариант: номер — целое число от 1 до 25, без кавычек"
CFG = VARIANTS[V]
EXCLUDE = CFG.get("exclude", [])
SCENARIO = ("прогноз после ПЕРВОГО периода: используется G1, G2 исключён" if "G2" in EXCLUDE
            else "прогноз после ВТОРОГО периода: используются G1 и G2")
print(f"Студент: {STUDENT_NAME}, группа {STUDENT_GROUP}")
print(f"Вариант {V}")
print("  признаки кластеризации:", ", ".join(CFG["features"]))
print("  цель:", "бинарный риск G3 < 10" if CFG["target"] == "binary" else "градация A/B/C/F")
print("  алгоритм:", CFG["algo"], CFG["params"])
print("  сценарий:", SCENARIO)
print("  педагогический кейс:", CFG["case"])

---
# Шаг 1. Загрузка данных по HTTPS

**Ячейка готова — просто запустите.** Код пробует адреса по очереди. Адрес из задания (`arunkumar-ramanan/...`) сейчас возвращает ошибку 404, поэтому автоматически используется резервная копия того же файла; третий вариант — официальный архив UCI.

`check_filled` — служебная функция: она останавливает ячейку с понятным сообщением, если вы забыли заменить `...` на код.

In [ ]:
import io, urllib.request, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25})

URLS = [
    "https://raw.githubusercontent.com/arunkumar-ramanan/student-performance-dataset/master/student-mat.csv",   # из задания
    "https://raw.githubusercontent.com/sachanganesh/student-performance-prediction/refs/heads/master/data/student-mat.csv",
    "https://archive.ics.uci.edu/static/public/320/student+performance.zip",                                   # официальный архив UCI
]

def read_bytes(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        return r.read()

def load_student_mat(urls):
    errors = []
    for url in urls:
        try:
            raw = read_bytes(url)
            if url.endswith(".zip"):                       # в архиве UCI ещё один архив student.zip
                outer = zipfile.ZipFile(io.BytesIO(raw))
                inner = zipfile.ZipFile(io.BytesIO(outer.read("student.zip")))
                raw = inner.read("student-mat.csv")
            frame = pd.read_csv(io.BytesIO(raw), sep=";")
            assert {"G1", "G2", "G3", "studytime"} <= set(frame.columns), "нет нужных столбцов"
            return frame, url
        except Exception as exc:
            errors.append(f"{url} → {type(exc).__name__}: {exc}")
    raise RuntimeError("Не удалось загрузить данные:\n" + "\n".join(errors))

def check_filled(where, values):
    """Проверяет, что вместо «...» вписан код; если нет — называет незаполненную строку."""
    empty = [name for name, v in values.items() if v is ...]
    if empty:
        raise ValueError(f"Не заполнено ({where}): " + ", ".join(empty)
                         + ". Замените «...» на код из подсказки «← впишите:» и запустите ячейку снова.")

def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:                                      # старые версии scikit-learn
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

df_raw, USED_URL = load_student_mat(URLS)
print("Источник:", USED_URL)
print("Размер:", df_raw.shape, "— строк (учащихся) и столбцов (признаков)")
df_raw.head()

**Как читать результат.** Одна строка — один учащийся, 33 столбца. Ключевые для нас:

| Признак | Смысл | Шкала |
|---|---|---|
| `G1`, `G2`, `G3` | оценки за 1-й, 2-й период и **итоговая** | 0–20, порог «сдал» — 10 |
| `studytime` | время на учёбу в неделю | 1: < 2 ч … 4: > 10 ч |
| `failures` | число прошлых неуспешных курсов | 0–3 (3 означает «3 и больше») |
| `absences` | число пропусков | 0–93 |
| `goout`, `Walc`, `famrel` | прогулки с друзьями, алкоголь в выходные, отношения в семье | 1 (мало/плохо) … 5 (много/отлично) |
| `higher`, `schoolsup`, `famsup`, `internet` | хочет высшее образование, школьная и семейная поддержка, интернет дома | yes / no |

Это **анкетные данные подростков**, собранные исследователями; многие признаки чувствительные (алкоголь, семья). Их можно использовать только для учебного анализа групп, а не для суждений о конкретном ученике.

---
# Шаг 2. Проверка качества данных

**Ячейка готова — просто запустите.** Проверяем пропуски, дубликаты и допустимые диапазоны оценок, затем создаём две цели: бинарный риск `risk` и градацию `grade`.

In [ ]:
print("Пропусков всего:", int(df_raw.isna().sum().sum()))
print("Полных дубликатов:", int(df_raw.duplicated().sum()))

grades = ["G1", "G2", "G3"]
display(pd.DataFrame({"min": df_raw[grades].min(), "max": df_raw[grades].max(),
                      "вне 0–20": ((df_raw[grades] < 0) | (df_raw[grades] > 20)).sum(),
                      "нулей": (df_raw[grades] == 0).sum()}))

df = df_raw.drop_duplicates().reset_index(drop=True).copy()
df["risk"] = (df["G3"] < 10).astype(int)
df["grade"] = pd.cut(df["G3"], bins=[-1, 9, 13, 15, 20], labels=["F", "C", "B", "A"]).astype(str)

print("\nБинарный риск (G3 < 10):")
display(df["risk"].value_counts().rename({0: "нет риска (0)", 1: "риск (1)"}).to_frame("учащихся")
        .assign(доля=lambda t: (t["учащихся"] / len(df)).round(3)))
print("Градация A/B/C/F:")
display(df["grade"].value_counts().reindex(["A", "B", "C", "F"]).to_frame("учащихся"))

**Как читать результат.**

* Пропусков и дубликатов нет — данные «чистые», но это нужно было **проверить**, а не предположить.
* Оценки лежат в диапазоне 0–20. Обратите внимание на **нули в `G2` и `G3`**: при `G1` > 0 итоговый 0 обычно означает, что ученик не пришёл на итоговую аттестацию, а не «знает на ноль». Для модели это всё равно риск, но для педагога — другая ситуация.
* Классы несбалансированы: учащихся без риска примерно вдвое больше. Поэтому дальше мы будем смотреть не только на Accuracy.

**Что получено.** Допишите:

> Пропусков — ___, дубликатов — ___. Оценки находятся в диапазоне ___–___; нулевых `G3` — ___.
> Учащихся с риском (`G3 < 10`) — ___ из ___ (___ %). Это важно для выбора метрики, потому что ___.

---
# Шаг 3. Подготовка признаков для кластеризации (TODO 1)

**Что делает ячейка.** Берёт признаки вашего варианта, сама делит их на числовые и категориальные и собирает «конвейер подготовки» `ColumnTransformer`: числовые → стандартизация, категориальные → One-Hot.

**Задание.** Допишите **две строки**:

* `scaler = StandardScaler()` — объект, который переведёт каждый числовой признак в шкалу $z=(x-\mu)/\sigma$;
* `encoder = make_ohe()` — объект One-Hot кодирования (функция `make_ohe` уже написана в шаге 1).

In [ ]:
cluster_features = CFG["features"]
cluster_categorical = [c for c in cluster_features if df[c].dtype == "object"]
cluster_numeric = [c for c in cluster_features if c not in cluster_categorical]
print("Числовые:", cluster_numeric)
print("Категориальные:", cluster_categorical or "нет")

scaler = ...     # TODO 1.1 ← впишите: StandardScaler()
encoder = ...    # TODO 1.2 ← впишите: make_ohe()
check_filled("шаг 3, TODO 1", {"scaler": scaler, "encoder": encoder})

cluster_preprocessor = ColumnTransformer(
    [("num", scaler, cluster_numeric), ("cat", encoder, cluster_categorical)],
    remainder="drop", verbose_feature_names_out=False)
X_cluster = cluster_preprocessor.fit_transform(df[cluster_features])
cluster_feature_names = list(cluster_preprocessor.get_feature_names_out())

print("\nМатрица для K-Means:", X_cluster.shape, "→", cluster_feature_names)
before_after = pd.DataFrame({"до: среднее": df[cluster_numeric].mean(), "до: σ": df[cluster_numeric].std(ddof=0),
                             "после: среднее": X_cluster[:, :len(cluster_numeric)].mean(0),
                             "после: σ": X_cluster[:, :len(cluster_numeric)].std(0)}, index=cluster_numeric)
display(before_after.round(2))

**Как читать результат.**

* Таблица «до / после»: после стандартизации у каждого числового признака среднее 0 и σ = 1. Теперь один «шаг» по `absences` весит столько же, сколько по `studytime`.
* Число столбцов матрицы больше числа признаков: каждый категориальный признак `yes`/`no` превратился в два столбца.

**Что получено.** Допишите:

> В варианте ___ числовых и ___ категориальных признаков; после подготовки матрица имеет размер ___ × ___.
> Без масштабирования K-Means сильнее всего «слушал» бы признак ___, потому что у него самая большая σ (___).

---
# Шаг 4. Сколько групп выделять: локоть и силуэт (TODO 2)

**Что делает ячейка.** Обучает K-Means для `k = 2…8` и для каждого `k` считает инерцию и средний силуэт, затем рисует оба графика рядом.

**Задание.** Посмотрите на графики и таблицу и **сами выберите** `k`. Впишите его в строку `BEST_K = ...`. Можно взять `k` с максимальным силуэтом (подсказка в строке) или другое значение, если оно лучше объясняется педагогически, — но тогда обоснуйте выбор в выводе.

In [ ]:
k_values = list(range(2, 9))
rows = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_cluster)
    rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(X_cluster, labels),
                 "размер наименьшей группы": int(np.bincount(labels).min())})
k_metrics = pd.DataFrame(rows)
display(k_metrics.round(4))

k_sil = int(k_metrics.loc[k_metrics["silhouette"].idxmax(), "k"])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(k_metrics["k"], k_metrics["inertia"], "o-"); ax[0].set(title="Метод локтя: инерция", xlabel="k", ylabel="inertia")
ax[1].plot(k_metrics["k"], k_metrics["silhouette"], "o-", color="tab:green"); ax[1].axvline(k_sil, ls="--", color="gray")
ax[1].set(title=f"Силуэт: максимум при k = {k_sil}", xlabel="k", ylabel="средний силуэт")
plt.tight_layout(); plt.show()

BEST_K = ...     # TODO 2 ← впишите выбранное число, например: k_sil  (k с максимальным силуэтом)
check_filled("шаг 4, TODO 2", {"BEST_K": BEST_K})
BEST_K = int(BEST_K); assert 2 <= BEST_K <= 8, "TODO 2: k должно быть от 2 до 8"
print("Выбрано k =", BEST_K)

**Как читать результат.**

* **Инерция** падает при любом росте `k` — это нормально. Ищите «локоть»: после него каждый новый кластер почти ничего не добавляет.
* **Силуэт** показывает, насколько группы отделены друг от друга. Значения 0,1–0,2 — типичная картина для анкет о людях: группы размыты, граница между ними условная. Это не ошибка, а свойство данных; называть кластеры «типами учеников» нельзя.
* Столбец «размер наименьшей группы»: группа из 10–15 учащихся неустойчива и плохо подходит для педагогических выводов.

**Что получено.** Допишите:

> Максимальный силуэт ___ получен при k = ___; «локоть» на графике инерции примерно при k = ___.
> Я выбрал(а) k = ___, потому что ___. Значение силуэта говорит о том, что группы ___ (чёткие / размытые).

---
# Шаг 5. Профили кластеров и группа повышенного внимания

**Ячейка готова — просто запустите.** Обучаем итоговый K-Means и описываем каждую группу **в исходных шкалах** (средние оценки, медиана пропусков, доли `yes`). Затем по учебным признакам выбираем группу повышенного педагогического внимания: низкие `G1`/`G2`, много неудач и пропусков.

Итоговая оценка `G3` в кластеризации **не участвовала**. Мы используем её только в конце, как внешнюю проверку: действительно ли в выбранной группе больше риска?

In [ ]:
kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=30)
df["cluster"] = kmeans.fit_predict(X_cluster)

prof = df.copy()
agg = {"n": ("G3", "size"), "G1_mean": ("G1", "mean"), "G2_mean": ("G2", "mean"),
       "failures_mean": ("failures", "mean"), "absences_median": ("absences", "median"),
       "studytime_mean": ("studytime", "mean")}
for c in cluster_numeric:
    if c not in ["G1", "G2", "failures", "absences", "studytime"]:
        agg[c + "_mean"] = (c, "mean")
for c in cluster_categorical:
    levels = sorted(prof[c].unique())
    if set(levels) <= {"yes", "no"}:
        prof[c + "_yes"] = prof[c].eq("yes").astype(int); agg[c + "_yes_share"] = (c + "_yes", "mean")
    else:
        for lv in levels:
            prof[f"{c}={lv}"] = prof[c].eq(lv).astype(int); agg[f"{c}={lv}_share"] = (f"{c}={lv}", "mean")
cluster_profile = prof.groupby("cluster").agg(**agg)

# Группа повышенного внимания: сумма рангов (низкие G1, G2; высокие failures, медиана absences)
ranks = pd.DataFrame({"G1": cluster_profile["G1_mean"].rank(method="min"),
                      "G2": cluster_profile["G2_mean"].rank(method="min"),
                      "failures": cluster_profile["failures_mean"].rank(ascending=False, method="min"),
                      "absences": cluster_profile["absences_median"].rank(ascending=False, method="min")})
cluster_profile["attention_score"] = ranks.sum(axis=1)
attention_cluster = int(cluster_profile["attention_score"].idxmin())

# Внешняя проверка — только после кластеризации
check = df.groupby("cluster").agg(G3_mean=("G3", "mean"), risk_share=("risk", "mean"),
                                  F_share=("grade", lambda s: (s == "F").mean()))
print("Профили кластеров (исходные шкалы):"); display(cluster_profile.round(2))
print(f"Группа повышенного внимания по учебным признакам: кластер {attention_cluster}")
print("Внешняя проверка по G3 (в кластеризации не использовалась):"); display(check.round(3))

show = cluster_profile[[c for c in cluster_profile.columns if c not in ["n", "attention_score"]]]
z = (show - show.mean()) / show.std(ddof=0).replace(0, 1)
plt.figure(figsize=(min(16, 1.1 * show.shape[1] + 3), 0.6 * BEST_K + 2))
sns.heatmap(z, annot=show.round(2), fmt="", cmap="RdBu_r", center=0, cbar_kws={"label": "отклонение от среднего по кластерам, σ"})
plt.title("Профили кластеров: цвет — выше/ниже среднего, числа — исходные значения"); plt.ylabel("кластер"); plt.grid(False); plt.tight_layout(); plt.show()

**Как читать результат.**

* Таблица профилей — «портрет» каждой группы: сколько в ней учащихся, какие у них средние оценки, сколько пропусков, какая доля хочет получать высшее образование и т. д.
* Тепловая карта: **красный** — показатель выше среднего по кластерам, **синий** — ниже; внутри клеток — исходные числа. Так удобно описать группу словами: «низкие текущие оценки, больше прошлых неудач и прогулок».
* Внешняя проверка: если в группе повышенного внимания доля `G3 < 10` заметно выше, чем в остальных, кластеризация уловила учебно значимое различие.
* Называйте группы **по наблюдаемым учебным показателям** («низкие текущие оценки и много пропусков»), а не ярлыками («слабые», «неблагополучные»).

**Что получено.** Допишите:

> Выделено ___ кластера(ов) размером ___. Группа повышенного внимания — кластер ___: средние G1 = ___, G2 = ___, неудач в среднем ___, медиана пропусков ___.
> По внешней проверке доля риска в этой группе ___ против ___ в остальных, то есть кластеризация ___ (согласуется / не согласуется) с итоговыми оценками.
> Педагогическое описание группы без ярлыков: ___.

---
# Шаг 6. Цель прогноза и разбиение на обучение и проверку (TODO 3–4)

**Что делает ячейка.** Формирует таблицу признаков `X` и цель `y` вашего варианта и делит данные на обучающую (75 %) и тестовую (25 %) части.

**Задание.** Допишите **две строки**:

* `X = df.drop(columns=["G3", "risk", "grade", "cluster"] + EXCLUDE)` — убрать из признаков **итоговую оценку и всё, что из неё получено**. Если оставить `G3`, модель «подсмотрит ответ» — это называется **утечкой данных**.
* `stratify=y` — стратифицированное разбиение: доля риска в train и test будет одинаковой.

In [ ]:
TARGET = "risk" if CFG["target"] == "binary" else "grade"
y = df[TARGET]

X = ...          # TODO 3 ← впишите: df.drop(columns=["G3", "risk", "grade", "cluster"] + EXCLUDE)
check_filled("шаг 6, TODO 3", {"X": X})
assert "G3" not in X.columns and TARGET not in X.columns, "TODO 3: в X не должно быть G3 и целевой переменной — это утечка ответа"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42,
    stratify=...,   # TODO 4 ← впишите: y
)
categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

print("Сценарий:", SCENARIO)
print(f"Признаков: {X.shape[1]} (числовых {len(numeric_features)}, категориальных {len(categorical_features)})")
print(f"Train: {len(X_train)} учащихся, Test: {len(X_test)} учащихся")
display(pd.DataFrame({"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)}).round(3))

**Как читать результат.**

* Таблица долей классов: в train и test они почти одинаковые — это и есть результат `stratify=y`. Без него в тест могло случайно попасть мало учащихся с риском, и оценка качества была бы ненадёжной.
* Сценарий: если в признаках есть `G2`, модель годится только **после второго периода** — раньше этих оценок просто нет. Для варианта 11 `G2` исключён, это сценарий раннего вмешательства после первого периода.

**Что получено.** Допишите:

> Из признаков исключены ___, потому что ___. Сценарий применения модели: ___.
> Train — ___ учащихся, test — ___; доля класса риска в train ___, в test ___.

---
# Шаг 7. Модель вашего варианта

**Ячейка готова — просто запустите.** Алгоритм и гиперпараметры берутся из таблицы вариантов. Модель собрана в `Pipeline`: подготовка признаков (масштаб + One-Hot) обучается **только на train**, затем — классификатор.

| Алгоритм | Как принимает решение | Для учителя |
|---|---|---|
| Decision Tree | цепочка вопросов «если G2 < 9,5, то…» | правила можно прочитать |
| Random Forest | голосование сотен деревьев | точнее одного дерева, но менее прозрачно |
| Logistic Regression | взвешенная сумма признаков → вероятность | веса показывают направление связи |
| KNN | смотрит на `k` самых похожих учеников | «у похожих учеников было так» |

`class_weight="balanced"` заставляет модель внимательнее относиться к редкому классу риска.

In [ ]:
ALGOS = {"Decision Tree": DecisionTreeClassifier, "Random Forest": RandomForestClassifier,
         "Logistic Regression": LogisticRegression, "KNN": KNeighborsClassifier}

def build_classifier(algo, params):
    p = dict(params)
    if algo in ("Decision Tree", "Random Forest", "Logistic Regression"):
        p.setdefault("random_state", 42)
    if algo == "Logistic Regression":
        p.setdefault("max_iter", 2000)
    if algo == "Random Forest":
        p.setdefault("n_jobs", -1)
    return ALGOS[algo](**p)

def make_model(num, cat):
    pre = ColumnTransformer([("num", StandardScaler(), num), ("cat", make_ohe(), cat)],
                            remainder="drop", verbose_feature_names_out=False)
    return Pipeline([("preprocess", pre), ("classifier", build_classifier(CFG["algo"], CFG["params"]))])

model = make_model(numeric_features, categorical_features)
model.fit(X_train, y_train)
print("Обучена модель:", model.named_steps["classifier"])

---
# Шаг 8. Оценка: матрица ошибок, F1 и сравнение с простыми правилами

**Ячейка готова — просто запустите.** Считаем метрики на тестовой части и сравниваем модель с двумя ориентирами:

* **«никто не рискует»** — всегда отвечать самым частым классом;
* **правило учителя** — та же шкала, применённая к последней известной оценке (`G2 < 10` или градация по `G2`; для варианта 11 — по `G1`).

Если модель не лучше правила учителя, её сложность не оправдана — это **нормальный и важный вывод**, а не провал.

In [ ]:
y_pred = model.predict(X_test)
RISK = 1 if TARGET == "risk" else "F"
labels = [0, 1] if TARGET == "risk" else ["F", "C", "B", "A"]
names = ["нет риска", "риск"] if TARGET == "risk" else labels
avg = "binary" if TARGET == "risk" else "macro"

def scores(y_true, y_hat):
    if TARGET == "risk":
        f1_main = f1_score(y_true, y_hat, pos_label=1, zero_division=0)
    else:
        f1_main = f1_score(y_true, y_hat, average="macro", zero_division=0)
    return {"Accuracy": accuracy_score(y_true, y_hat),
            "F1 (основная)": f1_main,
            "Precision риска": precision_score(y_true, y_hat, labels=[RISK], average="macro", zero_division=0),
            "Recall риска": recall_score(y_true, y_hat, labels=[RISK], average="macro", zero_division=0)}

last = "G1" if "G2" in EXCLUDE else "G2"
rule = (X_test[last] < 10).astype(int) if TARGET == "risk" else \
       pd.cut(X_test[last], bins=[-1, 9, 13, 15, 20], labels=["F", "C", "B", "A"]).astype(str)
majority = pd.Series([y_train.mode()[0]] * len(y_test), index=y_test.index)

compare = pd.DataFrame({f"Модель ({CFG['algo']})": scores(y_test, y_pred),
                        f"Правило учителя ({last})": scores(y_test, rule),
                        "«Никто не рискует»": scores(y_test, majority)}).T
print("Основная F1:", "F1 класса риска" if TARGET == "risk" else "macro-F1 по четырём классам")
display(compare.round(3))
print(classification_report(y_test, y_pred, labels=labels, target_names=names, digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(5.2 if TARGET == "risk" else 6, 4.4))
sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", xticklabels=names, yticklabels=names)
plt.xlabel("прогноз модели"); plt.ylabel("на самом деле"); plt.title("Матрица ошибок на тестовой части"); plt.grid(False); plt.tight_layout(); plt.show()
if TARGET == "risk":
    tn, fp, fn, tp = cm.ravel()
    print(f"TN = {tn}, FP = {fp} (лишние консультации), FN = {fn} (пропущенные ученики с риском), TP = {tp}")
else:
    fn = int(cm[0].sum() - cm[0, 0]); print(f"Учащихся с итогом F, которым модель поставила более высокий класс (аналог FN): {fn} из {int(cm[0].sum())}")

**Как читать результат.**

* **Матрица ошибок**: строка — как на самом деле, столбец — что предсказала модель. Диагональ — верные ответы. Для педагога главная клетка — «на самом деле риск, прогноз нет риска» (**FN**): эти ученики остались бы без помощи.
* **Recall риска** — какую долю учащихся с риском модель нашла; **Precision** — какая доля сигналов «риск» была настоящей.
* **Сравнение с правилом учителя**: если модель с десятками признаков почти не лучше правила «`G2` ниже 10», значит, её сила — в текущих оценках, а остальные признаки добавляют мало.
* **«Никто не рискует»** показывает, почему Accuracy обманчива: такая «модель» получает высокую Accuracy и нулевой Recall.

**Что получено.** Допишите:

> Основная F1 модели = ___, Accuracy = ___, Recall риска = ___, Precision риска = ___.
> Правило учителя по ___ даёт F1 = ___, «никто не рискует» — Accuracy ___ при Recall 0. Модель ___ (заметно лучше / сопоставима / хуже) правила, поэтому ___.
> Пропущено учащихся с риском (FN): ___; лишних сигналов (FP): ___. Для системы раннего предупреждения опаснее ___, потому что ___.

---
# Шаг 9. Какие признаки использует модель

**Ячейка готова — просто запустите.** Способ расчёта важности зависит от алгоритма:

* деревья и лес — `feature_importances_`: насколько признак помогал разделять классы;
* логистическая регрессия — модуль коэффициента (признаки масштабированы, поэтому сравнимы);
* KNN — перестановочная важность: насколько падает F1, если перемешать один признак в тесте.

Затем вторая проверка: переобучаем ту же модель **без социально-демографических признаков** (пол, возраст, адрес, семья, образование и работа родителей) и смотрим, изменилось ли качество.

In [ ]:
clf = model.named_steps["classifier"]
if hasattr(clf, "feature_importances_"):
    method = "feature_importances_ (вклад в разбиения деревьев)"
    imp = pd.Series(clf.feature_importances_, index=model.named_steps["preprocess"].get_feature_names_out())
elif hasattr(clf, "coef_"):
    method = "|коэффициент| логистической регрессии (признаки масштабированы)"
    imp = pd.Series(np.abs(clf.coef_).mean(axis=0), index=model.named_steps["preprocess"].get_feature_names_out())
else:
    method = "перестановочная важность: падение F1 при перемешивании признака"
    scoring = "f1" if TARGET == "risk" else "f1_macro"
    r = permutation_importance(model, X_test, y_test, scoring=scoring, n_repeats=10, random_state=42)
    imp = pd.Series(r.importances_mean, index=X_test.columns)
imp = imp.sort_values(ascending=False)
print("Метод:", method)
display(imp.head(10).round(4).to_frame("важность"))
imp.head(15).sort_values().plot.barh(figsize=(8, 5), color="tab:blue", title="15 самых важных признаков модели")
plt.xlabel("важность"); plt.tight_layout(); plt.show()

DEMOGRAPHIC = ["sex", "age", "address", "famsize", "Pstatus", "Medu", "Fedu", "Mjob", "Fjob", "guardian"]
keep = [c for c in X.columns if c not in DEMOGRAPHIC]
model_nd = make_model([c for c in numeric_features if c in keep], [c for c in categorical_features if c in keep])
model_nd.fit(X_train[keep], y_train)
f1_all = scores(y_test, y_pred)["F1 (основная)"]; f1_nd = scores(y_test, model_nd.predict(X_test[keep]))["F1 (основная)"]
print(f"Основная F1: со всеми признаками {f1_all:.3f}, без социально-демографических {f1_nd:.3f} (разница {f1_nd - f1_all:+.3f})")

**Как читать результат.**

* Если на первых местах `G2` и `G1`, модель в основном **продолжает текущие оценки**: кто учится слабо сейчас, рискует и в конце. Это ожидаемо, но означает, что модель мало говорит о причинах.
* Важность — это **вклад в решения этой модели**, а не причина неуспеваемости. Из того, что признак `goout` важен, нельзя сделать вывод «прогулки вызывают двойки».
* Проверка без социально-демографических признаков: если качество почти не изменилось, их можно и нужно не использовать — это снижает риск дискриминационных решений. Признаки вроде пола или образования родителей **нельзя** использовать для решений о конкретном ученике.

**Что получено.** Допишите:

> Самые важные признаки: ___, ___, ___. Это означает, что модель в основном опирается на ___.
> Важность не доказывает причину, потому что ___.
> Без социально-демографических признаков F1 = ___ (было ___), поэтому в школьной практике я бы ___ (оставил(а) / исключил(а)) их, потому что ___.

---
# Шаг 10. Педагогический кейс: индивидуальный образовательный маршрут

**Ячейка готова — просто запустите.** Формируется обезличенный реестр учащихся тестовой части, которых модель отнесла к риску, и для каждого — **черновик** мер поддержки по прозрачным правилам. Это предложение для учителя, а не решение.

```text
Прогноз риска → Проверка учителем → Диагностика → Персональные задания → Консультация
→ Восстановление пропущенных тем → Повторная диагностика через 2 недели → прогресс? да: снизить поддержку / нет: скорректировать ИОМ
```

In [ ]:
registry = X_test[[c for c in ["G1", "G2", "studytime", "failures", "absences", "schoolsup", "famsup", "higher", "internet"]
                   if c in X_test.columns]].copy()
registry["прогноз"] = y_pred
registry["вероятность риска"] = model.predict_proba(X_test)[:, list(model.classes_).index(RISK)].round(3)
registry["факт G3"] = df.loc[X_test.index, "G3"].values
registry.insert(0, "код", [f"S{i:03d}" for i in range(1, len(registry) + 1)])
group = registry[registry["прогноз"] == RISK].sort_values("вероятность риска", ascending=False)

def support_plan(r):
    a = []
    if r.get("G1", 20) < 10 or r.get("G2", 20) < 10: a.append("диагностическая работа и 2–3 задания по дефицитам")
    if r.get("failures", 0) > 0:                     a.append("еженедельная контрольная точка")
    if r.get("absences", 0) >= 10:                   a.append("план восстановления пропущенных тем")
    if r.get("schoolsup", "yes") == "no":            a.append("предложить школьную консультацию")
    a.append("решение учителя и повторная диагностика через 14 дней")
    return "; ".join(a)

group = group.assign(**{"черновик мер поддержки": group.apply(support_plan, axis=1)})
print(f"Модель отнесла к риску {len(group)} из {len(registry)} учащихся тестовой части; "
      f"из них фактически G3 < 10: {int((group['факт G3'] < 10).sum())}")
display(group.head(12))

**Как читать результат.** Реестр отсортирован по вероятности риска — это **порядок внимания учителя**, а не рейтинг учеников. Столбец «факт G3» есть только в учебной работе: в реальной школе его ещё нет, когда принимается решение. Строки с прогнозом «риск», но с фактом `G3 ≥ 10` — это ложные сигналы (FP): учитель потратит время на консультацию, но вреда ученику не будет, если поддержка не стигматизирует.

### ✍️ ИОМ для одного учащегося из реестра (заполните)

Выберите один код из реестра и заполните таблицу. Формулируйте измеримо и без ярлыков.

| Компонент ИОМ | Содержание |
|---|---|
| Учащийся (код) | _[например, S012]_ |
| Основание | _[какие учебные показатели требуют педагогической проверки: G1, G2, пропуски, неудачи]_ |
| Цель на 2 недели | _[конкретный результат: «решает задачи темы … не менее чем на 7 из 10»]_ |
| Диагностика | _[короткая работа, 10–15 минут, по каким темам]_ |
| Учебный контент | _[2–3 персональных задания]_ |
| Поддержка | _[консультация / работа в паре / цифровой ресурс]_ |
| Посещаемость | _[план восстановления пропущенных тем, если нужно]_ |
| Контроль | _[формирующее оценивание: что и когда]_ |
| Критерий прогресса | _[измеримое улучшение]_ |
| Пересмотр | _[через 14 дней: что делаем при прогрессе и без него]_ |

**Недопустимо:** автоматически снижать оценку, ограничивать доступ к сложным заданиям, использовать формулировки «неспособный», «слабый ученик», принимать решение по полу, семье или образованию родителей.

**Корректно (EN):** *The model indicates an elevated academic risk; the case should be reviewed by the teacher.*
**Некорректно:** *The model proves that the student is weak and will fail.*

---
# Шаг 11. Самопроверка

**Ячейка готова — просто запустите.** Все пункты должны быть `OK`.

In [ ]:
checks = {
    "Заполнены ФИО и группа":                       STUDENT_NAME != "Иванов Иван Иванович",
    "Данные загружены (395 строк)":                 len(df_raw) == 395,
    "Признаки масштабированы и закодированы":       X_cluster.shape[0] == len(df),
    "Выбрано k и обучен K-Means":                   df["cluster"].nunique() == BEST_K,
    "G3 исключён из признаков":                     "G3" not in X.columns,
    "Разбиение стратифицировано":                   (y_train.value_counts(normalize=True).sort_index() - y_test.value_counts(normalize=True).sort_index()).abs().max() < 0.03,
    "Посчитаны метрики и матрица ошибок":           cm.shape == (len(labels), len(labels)),
    "Посчитана важность признаков":                 len(imp) > 0,
    "Сформирован реестр для ИОМ":                   len(group) > 0,
}
for k, ok in checks.items():
    print(("OK " if ok else "НЕТ"), k)
print(f"\nВыполнено {sum(checks.values())} из {len(checks)}")

### ✍️ Итоговый вывод (6–8 предложений)

> 1) Какие группы выделил K-Means и насколько они чёткие (силуэт)? 2) Совпала ли группа повышенного внимания с фактическим риском? 3) Какова основная F1 модели и чем она отличается от правила учителя? 4) Сколько учащихся с риском модель пропустила? 5) На какие признаки опирается модель и почему это не причины? 6) Как вы используете прогноз в работе учителя и что модель делать не должна?
>
> _[ваш вывод]_